In [1]:
import os
import glob
import random
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix)
from scipy.io import loadmat
from scipy.stats import wilcoxon

def build_cnn(n_channels=19, n_timesamples=3840, fs=128):
    temporal_kernel_1 = int(fs / 4)
    temporal_kernel_2 = int(fs / 8)
    inp = layers.Input(shape=(n_channels, n_timesamples, 1), name="input_eeg")
    x = layers.Conv2D(16, (10, 1), padding="valid", activation="relu", name="spatial_conv1")(inp)
    x = layers.BatchNormalization(name="bn_spatial1")(x)
    x = layers.AveragePooling2D((2, 1), name="pool_spatial1")(x)
    x = layers.Conv2D(16, (4, 1), padding="valid", activation="relu", name="spatial_conv2")(x)
    x = layers.BatchNormalization(name="bn_spatial2")(x)
    x = layers.AveragePooling2D((2, 1), name="pool_spatial2")(x)
    x = layers.Reshape((x.shape[2], x.shape[3]), name="reshape_to_temporal")(x)
    x = layers.Conv1D(32, temporal_kernel_1, padding="valid", activation="relu", name="temporal_conv1")(x)
    x = layers.BatchNormalization(name="bn_temporal1")(x)
    x = layers.AveragePooling1D(temporal_kernel_1 // 2, name="pool_temporal1")(x)
    x = layers.Conv1D(32, temporal_kernel_2, padding="valid", activation="relu", name="temporal_conv2")(x)
    x = layers.BatchNormalization(name="bn_temporal2")(x)
    x = layers.AveragePooling1D(temporal_kernel_2 // 2, name="pool_temporal2")(x)
    flat = layers.Flatten(name="flatten")(x)
    dense1 = layers.Dense(64, activation="relu", name="dense1")(flat)
    dense2 = layers.Dense(32, activation="relu", name="dense2")(dense1)
    out = layers.Dense(1, activation="sigmoid", name="classification")(dense2)
    model = models.Model(inputs=inp, outputs=out, name="adhd_cnn")
    feature_model = models.Model(inputs=inp, outputs=[flat, dense1, dense2], name="adhd_cnn_features")
    return model, feature_model

fs_assumed = 128
epoch_seconds = 30
epoch_len = fs_assumed * epoch_seconds

base = "/kaggle/input/datasets/abinayajone/adhd-eeg-dataset"
folders = {"ADHD_part1": 1, "ADHD_part2": 1, "Control_part1": 0, "Control_part2": 0}

all_epochs, all_labels, all_subject_ids = [], [], []
for folder_name, label in folders.items():
    folder_path = f"{base}/{folder_name}/{folder_name}"
    files = sorted(glob.glob(folder_path + "/*.mat"))
    print(f"{folder_name}: {len(files)} files")
    for f in files:
        data = loadmat(f)
        key = [k for k in data.keys() if not k.startswith("__")][0]
        arr = data[key]
        n_epochs = arr.shape[0] // epoch_len
        subject_id = f.split("/")[-1].replace(".mat", "")
        for i in range(n_epochs):
            segment = arr[i*epoch_len:(i+1)*epoch_len, :]
            all_epochs.append(segment.T)
            all_labels.append(label)
            all_subject_ids.append(subject_id)

X = np.array(all_epochs)[..., np.newaxis]
y = np.array(all_labels)
groups = np.array(all_subject_ids)
print("X:", X.shape, " y:", y.shape, " unique subjects:", len(set(groups)))

def normalize_epoch(epoch):
    mean = epoch.mean(axis=1, keepdims=True)
    std = epoch.std(axis=1, keepdims=True)
    std[std == 0] = 1
    return (epoch - mean) / std

X_norm = np.array([normalize_epoch(e) for e in X])
print("X_norm ready:", X_norm.shape)

def compute_metrics(y_true, y_pred, y_proba):
    return {
        "accuracy": accuracy_score(y_true, y_pred) * 100,
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "auc": roc_auc_score(y_true, y_proba) if len(set(y_true)) > 1 else np.nan,
    }

def set_all_seeds(seed=42):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

set_all_seeds(42)
gkf = GroupKFold(n_splits=10)
fold_sizes = [len(fold_test_idx) for _, fold_test_idx in gkf.split(X_norm, y, groups)]
print("Fold sizes:", fold_sizes)
print("\nReload complete.")

ADHD_part1: 30 files
ADHD_part2: 31 files
Control_part1: 30 files
Control_part2: 30 files
X: (508, 19, 3840, 1)  y: (508,)  unique subjects: 121
X_norm ready: (508, 19, 3840, 1)
Fold sizes: [51, 51, 51, 51, 51, 51, 52, 50, 50, 50]

Reload complete.


In [2]:
import numpy as np

def coral_transform(source_features, target_features, lambda_reg=1e-3):
    """
    CORAL (CORrelation ALignment) feature transform.

    Aligns the second-order statistics (covariance) of source_features
    to match target_features, following Sun & Saenko (2016) -- I'm
    recalling the core mechanics from general domain-adaptation
    knowledge, not from a direct re-read of the paper just now, so
    please verify the exact formulation against the original paper
    before citing it precisely in your Methods section.

    Args:
        source_features: (n_source_samples, n_features) -- e.g., Dense1
                          features from the TRAINING folds' subjects
        target_features: (n_target_samples, n_features) -- Dense1
                          features from the TEST fold's subjects
        lambda_reg: small regularization term added to covariance
                    diagonals for numerical stability (matrix inversion
                    can be unstable otherwise)

    Returns:
        source_features_aligned: source features transformed to match
                                  target's covariance structure
    """
    # Center both feature sets
    source_centered = source_features - source_features.mean(axis=0, keepdims=True)
    target_centered = target_features - target_features.mean(axis=0, keepdims=True)

    n_s = source_features.shape[0]
    n_t = target_features.shape[0]
    d = source_features.shape[1]

    # Covariance matrices (with regularization for numerical stability)
    cov_source = (source_centered.T @ source_centered) / (n_s - 1) + lambda_reg * np.eye(d)
    cov_target = (target_centered.T @ target_centered) / (n_t - 1) + lambda_reg * np.eye(d)

    # Whitening transform using source covariance, then re-coloring with target covariance
    # A_source = C_source^(-1/2), A_target = C_target^(1/2)
    # source_aligned = source_centered @ A_source @ A_target

    # Matrix square root / inverse square root via eigendecomposition
    def matrix_power(mat, power):
        eigvals, eigvecs = np.linalg.eigh(mat)
        eigvals = np.clip(eigvals, a_min=1e-12, a_max=None)  # avoid negative/zero eigenvalues
        return eigvecs @ np.diag(eigvals ** power) @ eigvecs.T

    cov_source_inv_sqrt = matrix_power(cov_source, -0.5)
    cov_target_sqrt = matrix_power(cov_target, 0.5)

    source_aligned = source_centered @ cov_source_inv_sqrt @ cov_target_sqrt

    # Re-add target's mean so aligned source features sit in target's location too
    source_aligned = source_aligned + target_features.mean(axis=0, keepdims=True)

    return source_aligned


# --- Quick sanity check on synthetic data before touching real features ---
np.random.seed(0)
fake_source = np.random.randn(100, 64) * 3 + 5   # different scale/shift than target
fake_target = np.random.randn(50, 64) * 1 + 0

print("Before CORAL:")
print("  Source mean/std:", fake_source.mean(), fake_source.std())
print("  Target mean/std:", fake_target.mean(), fake_target.std())

fake_source_aligned = coral_transform(fake_source, fake_target)

print("\nAfter CORAL (source aligned to target):")
print("  Aligned source mean/std:", fake_source_aligned.mean(), fake_source_aligned.std())
print("  Target mean/std (unchanged):", fake_target.mean(), fake_target.std())

Before CORAL:
  Source mean/std: 4.956216435436183 2.9631683646583715
  Target mean/std: -0.03143912335870829 0.9871141132614175

After CORAL (source aligned to target):
  Aligned source mean/std: -0.03143912335870838 0.9923793865586367
  Target mean/std (unchanged): -0.03143912335870829 0.9871141132614175


In [3]:
# More rigorous check: does the FULL covariance matrix align, not just mean/std?
def covariance_distance(feat_a, feat_b):
    """Frobenius norm of the difference between two covariance matrices --
    smaller means more similar covariance structure."""
    cov_a = np.cov(feat_a, rowvar=False)
    cov_b = np.cov(feat_b, rowvar=False)
    return np.linalg.norm(cov_a - cov_b, ord="fro")

dist_before = covariance_distance(fake_source, fake_target)
dist_after = covariance_distance(fake_source_aligned, fake_target)

print(f"Covariance distance BEFORE CORAL: {dist_before:.4f}")
print(f"Covariance distance AFTER CORAL:  {dist_after:.4f}")
print(f"Reduction: {(1 - dist_after/dist_before)*100:.1f}%")

Covariance distance BEFORE CORAL: 84.1653
Covariance distance AFTER CORAL:  0.0071
Reduction: 100.0%


In [4]:
# Test CORAL on a single real fold, using fold 1 from our GroupKFold split
fold_iter = gkf.split(X_norm, y, groups)
fold_train_idx, fold_test_idx = next(fold_iter)  # just the first fold

X_fold_train, X_fold_test = X_norm[fold_train_idx], X_norm[fold_test_idx]
y_fold_train, y_fold_test = y[fold_train_idx], y[fold_test_idx]

early_stop_test = EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True)
model_test, feature_model_test = build_cnn(n_channels=19, n_timesamples=3840, fs=128)
model_test.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
                    loss="binary_crossentropy", metrics=["accuracy"])
model_test.fit(X_fold_train, y_fold_train, epochs=100, batch_size=16, verbose=0,
                validation_data=(X_fold_test, y_fold_test), callbacks=[early_stop_test])

train_feats_test = feature_model_test.predict(X_fold_train, verbose=0)[1]  # Dense1, (n_train, 64)
test_feats_test = feature_model_test.predict(X_fold_test, verbose=0)[1]    # Dense1, (n_test, 64)

print("Before CORAL - covariance distance (train vs test features):",
      covariance_distance(train_feats_test, test_feats_test))

train_feats_aligned = coral_transform(train_feats_test, test_feats_test)

print("After CORAL  - covariance distance (aligned train vs test features):",
      covariance_distance(train_feats_aligned, test_feats_test))

# Train LR on ALIGNED source features, evaluate on (unchanged) target features
lr_coral = LogisticRegression(penalty="l2", max_iter=2000, class_weight="balanced")
lr_coral.fit(train_feats_aligned, y_fold_train)
pred_coral = lr_coral.predict(test_feats_test)
proba_coral = lr_coral.predict_proba(test_feats_test)[:, 1]

# Compare to LR WITHOUT CORAL (same fold, unaligned features)
lr_no_coral = LogisticRegression(penalty="l2", max_iter=2000, class_weight="balanced")
lr_no_coral.fit(train_feats_test, y_fold_train)
pred_no_coral = lr_no_coral.predict(test_feats_test)
proba_no_coral = lr_no_coral.predict_proba(test_feats_test)[:, 1]

print("\nFold 1 -- LR without CORAL:", compute_metrics(y_fold_test, pred_no_coral, proba_no_coral))
print("Fold 1 -- LR with CORAL:   ", compute_metrics(y_fold_test, pred_coral, proba_coral))

I0000 00:00:1783243872.027641      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1783243872.033607      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
I0000 00:00:1783243883.091417     130 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


Before CORAL - covariance distance (train vs test features): 0.00921544570098638
After CORAL  - covariance distance (aligned train vs test features): 0.0039039059245574055

Fold 1 -- LR without CORAL: {'accuracy': 45.09803921568628, 'precision': 0.5555555555555556, 'recall': 0.4838709677419355, 'f1': 0.5172413793103449, 'auc': np.float64(0.4854838709677419)}
Fold 1 -- LR with CORAL:    {'accuracy': 52.94117647058824, 'precision': 0.6, 'recall': 0.6774193548387096, 'f1': 0.6363636363636364, 'auc': np.float64(0.48225806451612907)}


In [12]:
import pickle

checkpoint_coral_path = "/kaggle/working/fold_results_coral_comparison.pkl"

results_coral = {"no_coral": {"y_true": [], "pred": [], "proba": []},
                  "with_coral": {"y_true": [], "pred": [], "proba": []}}

set_all_seeds(42)

for fold_i, (fold_train_idx, fold_test_idx) in enumerate(gkf.split(X_norm, y, groups)):
    X_fold_train, X_fold_test = X_norm[fold_train_idx], X_norm[fold_test_idx]
    y_fold_train, y_fold_test = y[fold_train_idx], y[fold_test_idx]

    early_stop_fold = EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True)
    model_fold, feature_model_fold = build_cnn(n_channels=19, n_timesamples=3840, fs=128)
    model_fold.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
                        loss="binary_crossentropy", metrics=["accuracy"])
    model_fold.fit(X_fold_train, y_fold_train, epochs=100, batch_size=16, verbose=0,
                    validation_data=(X_fold_test, y_fold_test), callbacks=[early_stop_fold])

    train_feats = feature_model_fold.predict(X_fold_train, verbose=0)[1]
    test_feats = feature_model_fold.predict(X_fold_test, verbose=0)[1]

    # --- No CORAL ---
    lr_no_coral = LogisticRegression(penalty="l2", max_iter=2000, class_weight="balanced")
    lr_no_coral.fit(train_feats, y_fold_train)
    pred_nc = lr_no_coral.predict(test_feats)
    proba_nc = lr_no_coral.predict_proba(test_feats)[:, 1]
    results_coral["no_coral"]["y_true"].extend(y_fold_test)
    results_coral["no_coral"]["pred"].extend(pred_nc)
    results_coral["no_coral"]["proba"].extend(proba_nc)

    # --- With CORAL ---
    train_feats_aligned = coral_transform(train_feats, test_feats)
    lr_coral = LogisticRegression(penalty="l2", max_iter=2000, class_weight="balanced")
    lr_coral.fit(train_feats_aligned, y_fold_train)
    pred_c = lr_coral.predict(test_feats)
    proba_c = lr_coral.predict_proba(test_feats)[:, 1]
    results_coral["with_coral"]["y_true"].extend(y_fold_test)
    results_coral["with_coral"]["pred"].extend(pred_c)
    results_coral["with_coral"]["proba"].extend(proba_c)

    with open(checkpoint_coral_path, "wb") as fh:
        pickle.dump(results_coral, fh)

    print(f"Fold {fold_i+1}/10 done, checkpoint saved.")

print("\n=== POOLED metrics: CORAL vs no-CORAL ===")
for name in ["no_coral", "with_coral"]:
    y_true_arr = np.array(results_coral[name]["y_true"])
    pred_arr = np.array(results_coral[name]["pred"])
    proba_arr = np.array(results_coral[name]["proba"])
    m = compute_metrics(y_true_arr, pred_arr, proba_arr)
    print(f"{name}: {m}")

I0000 00:00:1783255944.864057      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1783255944.867108      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
I0000 00:00:1783255955.413764     136 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


Fold 1/10 done, checkpoint saved.
Fold 2/10 done, checkpoint saved.
Fold 3/10 done, checkpoint saved.
Fold 4/10 done, checkpoint saved.
Fold 5/10 done, checkpoint saved.
Fold 6/10 done, checkpoint saved.
Fold 7/10 done, checkpoint saved.
Fold 8/10 done, checkpoint saved.
Fold 9/10 done, checkpoint saved.
Fold 10/10 done, checkpoint saved.

=== POOLED metrics: CORAL vs no-CORAL ===
no_coral: {'accuracy': 66.14173228346458, 'precision': 0.6986301369863014, 'recall': 0.7083333333333334, 'f1': 0.7034482758620689, 'auc': np.float64(0.7462279040404041)}
with_coral: {'accuracy': 64.37007874015748, 'precision': 0.6945454545454546, 'recall': 0.6631944444444444, 'f1': 0.6785079928952042, 'auc': np.float64(0.727935606060606)}


In [10]:
def coral_transform(source_features, target_features, lambda_reg=1e-3):
    source_centered = source_features - source_features.mean(axis=0, keepdims=True)
    target_centered = target_features - target_features.mean(axis=0, keepdims=True)
    n_s, n_t, d = source_features.shape[0], target_features.shape[0], source_features.shape[1]
    cov_source = (source_centered.T @ source_centered) / (n_s - 1) + lambda_reg * np.eye(d)
    cov_target = (target_centered.T @ target_centered) / (n_t - 1) + lambda_reg * np.eye(d)

    def matrix_power(mat, power):
        eigvals, eigvecs = np.linalg.eigh(mat)
        eigvals = np.clip(eigvals, a_min=1e-12, a_max=None)
        return eigvecs @ np.diag(eigvals ** power) @ eigvecs.T

    cov_source_inv_sqrt = matrix_power(cov_source, -0.5)
    cov_target_sqrt = matrix_power(cov_target, 0.5)
    source_aligned = source_centered @ cov_source_inv_sqrt @ cov_target_sqrt
    source_aligned = source_aligned + target_features.mean(axis=0, keepdims=True)
    return source_aligned

In [13]:
no_coral_accs = []
with_coral_accs = []

start = 0
for size in fold_sizes:  # reuse fold_sizes from earlier, if still in memory; otherwise recompute
    end = start + size
    y_chunk = np.array(results_coral["no_coral"]["y_true"])[start:end]
    pred_nc_chunk = np.array(results_coral["no_coral"]["pred"])[start:end]
    pred_c_chunk = np.array(results_coral["with_coral"]["pred"])[start:end]
    no_coral_accs.append(accuracy_score(y_chunk, pred_nc_chunk) * 100)
    with_coral_accs.append(accuracy_score(y_chunk, pred_c_chunk) * 100)
    start = end

stat, p = wilcoxon(no_coral_accs, with_coral_accs)
print(f"No-CORAL fold accuracies: {no_coral_accs}")
print(f"With-CORAL fold accuracies: {with_coral_accs}")
print(f"Wilcoxon p-value: {p:.4f}")

No-CORAL fold accuracies: [43.13725490196079, 92.15686274509804, 90.19607843137256, 60.78431372549019, 64.70588235294117, 45.09803921568628, 61.53846153846154, 52.0, 84.0, 68.0]
With-CORAL fold accuracies: [52.94117647058824, 92.15686274509804, 86.27450980392157, 58.82352941176471, 56.86274509803921, 45.09803921568628, 59.61538461538461, 44.0, 80.0, 68.0]
Wilcoxon p-value: 0.2969


In [3]:
# --- Train one CNN on the FULL dataset, purely to extract filter weights
# for visualization (like base paper's Figs 4-9). This is NOT a
# performance evaluation -- no held-out test set here, since the only
# goal is representative trained filter shapes. ---

model_viz, feature_model_viz = build_cnn(n_channels=19, n_timesamples=3840, fs=128)
model_viz.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
                   loss="binary_crossentropy", metrics=["accuracy"])

model_viz.fit(X_norm, y, epochs=30, batch_size=16, verbose=1)

# --- Extract weights from each conv layer ---
spatial1_weights = model_viz.get_layer("spatial_conv1").get_weights()[0]  # shape (10, 1, 1, 16)
spatial2_weights = model_viz.get_layer("spatial_conv2").get_weights()[0]  # shape (4, 1, 16, 16)
temporal1_weights = model_viz.get_layer("temporal_conv1").get_weights()[0]  # shape (32, 16, 32)
temporal2_weights = model_viz.get_layer("temporal_conv2").get_weights()[0]  # shape (16, 32, 32)

print("spatial1_weights shape:", spatial1_weights.shape)
print("spatial2_weights shape:", spatial2_weights.shape)
print("temporal1_weights shape:", temporal1_weights.shape)
print("temporal2_weights shape:", temporal2_weights.shape)

# Save raw weights so we don't need to retrain if the session drops again
np.savez("/kaggle/working/trained_filter_weights.npz",
         spatial1=spatial1_weights, spatial2=spatial2_weights,
         temporal1=temporal1_weights, temporal2=temporal2_weights)
print("\nWeights saved to /kaggle/working/trained_filter_weights.npz")
print("Please download this file now, in case the session drops.")

I0000 00:00:1783266844.593800      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1783266844.596873      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Epoch 1/30
 1/32 ━━━━━━━━━━━━━━━━━━━━ 4:49 9s/step - accuracy: 0.4375 - loss: 0.7763

I0000 00:00:1783266855.657385     133 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


32/32 ━━━━━━━━━━━━━━━━━━━━ 16s 202ms/step - accuracy: 0.6102 - loss: 0.6761
Epoch 2/30
32/32 ━━━━━━━━━━━━━━━━━━━━ 2s 59ms/step - accuracy: 0.7736 - loss: 0.5588
Epoch 3/30
32/32 ━━━━━━━━━━━━━━━━━━━━ 2s 57ms/step - accuracy: 0.8445 - loss: 0.4725
Epoch 4/30
32/32 ━━━━━━━━━━━━━━━━━━━━ 2s 58ms/step - accuracy: 0.8720 - loss: 0.4005
Epoch 5/30
32/32 ━━━━━━━━━━━━━━━━━━━━ 2s 57ms/step - accuracy: 0.8937 - loss: 0.3442
Epoch 6/30
32/32 ━━━━━━━━━━━━━━━━━━━━ 2s 57ms/step - accuracy: 0.9035 - loss: 0.2977
Epoch 7/30
32/32 ━━━━━━━━━━━━━━━━━━━━ 2s 57ms/step - accuracy: 0.9252 - loss: 0.2565
Epoch 8/30
32/32 ━━━━━━━━━━━━━━━━━━━━ 2s 57ms/step - accuracy: 0.9409 - loss: 0.2203
Epoch 9/30
32/32 ━━━━━━━━━━━━━━━━━━━━ 2s 57ms/step - accuracy: 0.9528 - loss: 0.1894
Epoch 10/30
32/32 ━━━━━━━━━━━━━━━━━━━━ 2s 57ms/step - accuracy: 0.9626 - loss: 0.1627
Epoch 11/30
32/32 ━━━━━━━━━━━━━━━━━━━━ 2s 58ms/step - accuracy: 0.9705 - loss: 0.1394
Epoch 12/30
32/32 ━━━━━━━━━━━━━━━━━━━━ 2s 57ms/step - accuracy: 0.9764 -

In [7]:
import pickle
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier

checkpoint_path_all = "/kaggle/working/fold_results_all_classifiers.pkl"
classifier_names = ["CNN", "LR", "NLSVM", "RF", "GNB", "KNN"]
all_results = {name: {"y_true": [], "pred": [], "proba": []} for name in classifier_names}

set_all_seeds(42)

for fold_i, (fold_train_idx, fold_test_idx) in enumerate(gkf.split(X_norm, y, groups)):
    X_fold_train, X_fold_test = X_norm[fold_train_idx], X_norm[fold_test_idx]
    y_fold_train, y_fold_test = y[fold_train_idx], y[fold_test_idx]

    early_stop_fold = EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True)
    model_fold, feature_model_fold = build_cnn(n_channels=19, n_timesamples=3840, fs=128)
    model_fold.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
                        loss="binary_crossentropy", metrics=["accuracy"])
    model_fold.fit(X_fold_train, y_fold_train, epochs=100, batch_size=16, verbose=0,
                    validation_data=(X_fold_test, y_fold_test), callbacks=[early_stop_fold])

    proba_cnn = model_fold.predict(X_fold_test, verbose=0).ravel()
    pred_cnn = (proba_cnn >= 0.5).astype(int)
    all_results["CNN"]["y_true"].extend(y_fold_test)
    all_results["CNN"]["pred"].extend(pred_cnn)
    all_results["CNN"]["proba"].extend(proba_cnn)

    train_feats = feature_model_fold.predict(X_fold_train, verbose=0)[1]
    test_feats = feature_model_fold.predict(X_fold_test, verbose=0)[1]

    classifiers = {
        "LR": LogisticRegression(penalty="l2", max_iter=2000, class_weight="balanced"),
        "NLSVM": SVC(kernel="rbf", probability=True, class_weight="balanced"),
        "RF": RandomForestClassifier(n_estimators=100, class_weight="balanced"),
        "GNB": GaussianNB(),
        "KNN": KNeighborsClassifier(n_neighbors=5),
    }

    for name, clf in classifiers.items():
        clf.fit(train_feats, y_fold_train)
        pred = clf.predict(test_feats)
        proba = clf.predict_proba(test_feats)[:, 1] if hasattr(clf, "predict_proba") else pred
        all_results[name]["y_true"].extend(y_fold_test)
        all_results[name]["pred"].extend(pred)
        all_results[name]["proba"].extend(proba)

    with open(checkpoint_path_all, "wb") as fh:
        pickle.dump(all_results, fh)

    print(f"Fold {fold_i+1}/10 done, checkpoint saved.")

print("\nAll folds complete.")

Fold 1/10 done, checkpoint saved.
Fold 2/10 done, checkpoint saved.
Fold 3/10 done, checkpoint saved.
Fold 4/10 done, checkpoint saved.
Fold 5/10 done, checkpoint saved.
Fold 6/10 done, checkpoint saved.
Fold 7/10 done, checkpoint saved.
Fold 8/10 done, checkpoint saved.
Fold 9/10 done, checkpoint saved.
Fold 10/10 done, checkpoint saved.

All folds complete.


In [8]:
import numpy as np

adhd_idx = np.where(y == 1)[0][0]
control_idx = np.where(y == 0)[0][0]

np.savez("/kaggle/working/figure_data_export.npz",
         cnn_y_true=np.array(all_results["CNN"]["y_true"]),
         cnn_proba=np.array(all_results["CNN"]["proba"]),
         lr_proba=np.array(all_results["LR"]["proba"]),
         nlsvm_proba=np.array(all_results["NLSVM"]["proba"]),
         rf_proba=np.array(all_results["RF"]["proba"]),
         gnb_proba=np.array(all_results["GNB"]["proba"]),
         knn_proba=np.array(all_results["KNN"]["proba"]),
         adhd_epoch_raw=X[adhd_idx, :, :, 0],
         adhd_epoch_norm=X_norm[adhd_idx, :, :, 0],
         control_epoch_raw=X[control_idx, :, :, 0],
         control_epoch_norm=X_norm[control_idx, :, :, 0],
         )
print("Saved figure_data_export.npz -- please download and upload here immediately after this completes.")

Saved figure_data_export.npz -- please download and upload here immediately after this completes.


In [9]:
import pickle
from sklearn.model_selection import GroupShuffleSplit

# Single grouped 80/20 split -- same pipeline as our 10-fold run, just one split
splitter_single = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
single_train_idx, single_test_idx = next(splitter_single.split(X_norm, y, groups))

X_single_train, X_single_test = X_norm[single_train_idx], X_norm[single_test_idx]
y_single_train, y_single_test = y[single_train_idx], y[single_test_idx]

print(f"Single split -- train: {len(single_train_idx)} epochs, test: {len(single_test_idx)} epochs")
print(f"Test subjects: {len(set(groups[single_test_idx]))}")

single_split_results = {name: {} for name in classifier_names}

set_all_seeds(42)

early_stop_single = EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True)
model_single, feature_model_single = build_cnn(n_channels=19, n_timesamples=3840, fs=128)
model_single.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
                      loss="binary_crossentropy", metrics=["accuracy"])
model_single.fit(X_single_train, y_single_train, epochs=100, batch_size=16, verbose=0,
                  validation_data=(X_single_test, y_single_test), callbacks=[early_stop_single])

proba_cnn_single = model_single.predict(X_single_test, verbose=0).ravel()
pred_cnn_single = (proba_cnn_single >= 0.5).astype(int)
single_split_results["CNN"] = compute_metrics(y_single_test, pred_cnn_single, proba_cnn_single)

train_feats_single = feature_model_single.predict(X_single_train, verbose=0)[1]
test_feats_single = feature_model_single.predict(X_single_test, verbose=0)[1]

classifiers_single = {
    "LR": LogisticRegression(penalty="l2", max_iter=2000, class_weight="balanced"),
    "NLSVM": SVC(kernel="rbf", probability=True, class_weight="balanced"),
    "RF": RandomForestClassifier(n_estimators=100, class_weight="balanced"),
    "GNB": GaussianNB(),
    "KNN": KNeighborsClassifier(n_neighbors=5),
}

for name, clf in classifiers_single.items():
    clf.fit(train_feats_single, y_single_train)
    pred = clf.predict(test_feats_single)
    proba = clf.predict_proba(test_feats_single)[:, 1] if hasattr(clf, "predict_proba") else pred
    single_split_results[name] = compute_metrics(y_single_test, pred, proba)

with open("/kaggle/working/single_split_results.pkl", "wb") as fh:
    pickle.dump(single_split_results, fh)

print("\n=== Single-split results (our pipeline, fs=128Hz) ===")
for name in classifier_names:
    print(f"{name}: {single_split_results[name]}")

print("\nPlease download single_split_results.pkl now.")

Single split -- train: 404 epochs, test: 104 epochs
Test subjects: 25

=== Single-split results (our pipeline, fs=128Hz) ===
CNN: {'accuracy': 34.61538461538461, 'precision': 0.4090909090909091, 'recall': 0.140625, 'f1': 0.20930232558139536, 'auc': np.float64(0.36875)}
LR: {'accuracy': 60.57692307692307, 'precision': 0.6666666666666666, 'recall': 0.71875, 'f1': 0.6917293233082706, 'auc': np.float64(0.642578125)}
NLSVM: {'accuracy': 63.46153846153846, 'precision': 0.6805555555555556, 'recall': 0.765625, 'f1': 0.7205882352941176, 'auc': np.float64(0.637890625)}
RF: {'accuracy': 61.53846153846154, 'precision': 0.6578947368421053, 'recall': 0.78125, 'f1': 0.7142857142857143, 'auc': np.float64(0.6708984375)}
GNB: {'accuracy': 66.34615384615384, 'precision': 0.6559139784946236, 'recall': 0.953125, 'f1': 0.7770700636942676, 'auc': np.float64(0.67578125)}
KNN: {'accuracy': 51.92307692307693, 'precision': 0.5921052631578947, 'recall': 0.703125, 'f1': 0.6428571428571429, 'auc': np.float64(0.4785

In [2]:
import numpy as np

n_repeats = 20
single_split_accuracies = {name: [] for name in ["CNN", "LR", "NLSVM", "RF", "GNB", "KNN"]}

for repeat_i in range(n_repeats):
    splitter_repeat = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=repeat_i)
    train_idx_r, test_idx_r = next(splitter_repeat.split(X_norm, y, groups))

    X_train_r, X_test_r = X_norm[train_idx_r], X_norm[test_idx_r]
    y_train_r, y_test_r = y[train_idx_r], y[test_idx_r]

    early_stop_r = EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True)
    model_r, feature_model_r = build_cnn(n_channels=19, n_timesamples=3840, fs=128)
    model_r.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
                     loss="binary_crossentropy", metrics=["accuracy"])
    model_r.fit(X_train_r, y_train_r, epochs=100, batch_size=16, verbose=0,
                validation_data=(X_test_r, y_test_r), callbacks=[early_stop_r])

    proba_cnn_r = model_r.predict(X_test_r, verbose=0).ravel()
    pred_cnn_r = (proba_cnn_r >= 0.5).astype(int)
    single_split_accuracies["CNN"].append(accuracy_score(y_test_r, pred_cnn_r) * 100)

    train_feats_r = feature_model_r.predict(X_train_r, verbose=0)[1]
    test_feats_r = feature_model_r.predict(X_test_r, verbose=0)[1]

    classifiers_r = {
        "LR": LogisticRegression(penalty="l2", max_iter=2000, class_weight="balanced"),
        "NLSVM": SVC(kernel="rbf", probability=True, class_weight="balanced"),
        "RF": RandomForestClassifier(n_estimators=100, class_weight="balanced"),
        "GNB": GaussianNB(),
        "KNN": KNeighborsClassifier(n_neighbors=5),
    }
    for name, clf in classifiers_r.items():
        clf.fit(train_feats_r, y_train_r)
        pred_r = clf.predict(test_feats_r)
        single_split_accuracies[name].append(accuracy_score(y_test_r, pred_r) * 100)

    print(f"Repeat {repeat_i+1}/{n_repeats}: CNN={single_split_accuracies['CNN'][-1]:.1f}%, "
          f"LR={single_split_accuracies['LR'][-1]:.1f}%")

import pickle
with open("/kaggle/working/repeated_single_splits.pkl", "wb") as fh:
    pickle.dump(single_split_accuracies, fh)
print("\nSaved. Please download immediately.")

I0000 00:00:1783269443.201549      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1783269443.204643      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
I0000 00:00:1783269453.390721     133 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


Repeat 1/20: CNN=38.8%, LR=61.2%
Repeat 2/20: CNN=55.6%, LR=55.6%
Repeat 3/20: CNN=43.1%, LR=53.9%
Repeat 4/20: CNN=80.6%, LR=76.7%
Repeat 5/20: CNN=31.1%, LR=52.4%
Repeat 6/20: CNN=61.1%, LR=64.6%
Repeat 7/20: CNN=31.9%, LR=55.8%
Repeat 8/20: CNN=39.4%, LR=45.2%
Repeat 9/20: CNN=65.7%, LR=70.6%
Repeat 10/20: CNN=52.1%, LR=68.1%
Repeat 11/20: CNN=58.3%, LR=61.2%
Repeat 12/20: CNN=55.8%, LR=49.6%
Repeat 13/20: CNN=91.5%, LR=91.5%
Repeat 14/20: CNN=40.4%, LR=56.0%
Repeat 15/20: CNN=64.9%, LR=64.9%
Repeat 16/20: CNN=69.6%, LR=74.5%
Repeat 17/20: CNN=36.8%, LR=44.2%
Repeat 18/20: CNN=82.3%, LR=78.1%
Repeat 19/20: CNN=67.0%, LR=69.8%
Repeat 20/20: CNN=65.7%, LR=66.7%

Saved. Please download immediately.


In [3]:
import numpy as np
import pickle

n_repeats = 50
single_split_accuracies_50 = {name: [] for name in ["CNN", "LR", "NLSVM", "RF", "GNB", "KNN"]}

for repeat_i in range(n_repeats):
    splitter_repeat = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=repeat_i)
    train_idx_r, test_idx_r = next(splitter_repeat.split(X_norm, y, groups))

    X_train_r, X_test_r = X_norm[train_idx_r], X_norm[test_idx_r]
    y_train_r, y_test_r = y[train_idx_r], y[test_idx_r]

    early_stop_r = EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True)
    model_r, feature_model_r = build_cnn(n_channels=19, n_timesamples=3840, fs=128)
    model_r.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
                     loss="binary_crossentropy", metrics=["accuracy"])
    model_r.fit(X_train_r, y_train_r, epochs=100, batch_size=16, verbose=0,
                validation_data=(X_test_r, y_test_r), callbacks=[early_stop_r])

    proba_cnn_r = model_r.predict(X_test_r, verbose=0).ravel()
    pred_cnn_r = (proba_cnn_r >= 0.5).astype(int)
    single_split_accuracies_50["CNN"].append(accuracy_score(y_test_r, pred_cnn_r) * 100)

    train_feats_r = feature_model_r.predict(X_train_r, verbose=0)[1]
    test_feats_r = feature_model_r.predict(X_test_r, verbose=0)[1]

    classifiers_r = {
        "LR": LogisticRegression(penalty="l2", max_iter=2000, class_weight="balanced"),
        "NLSVM": SVC(kernel="rbf", probability=True, class_weight="balanced"),
        "RF": RandomForestClassifier(n_estimators=100, class_weight="balanced"),
        "GNB": GaussianNB(),
        "KNN": KNeighborsClassifier(n_neighbors=5),
    }
    for name, clf in classifiers_r.items():
        clf.fit(train_feats_r, y_train_r)
        pred_r = clf.predict(test_feats_r)
        single_split_accuracies_50[name].append(accuracy_score(y_test_r, pred_r) * 100)

    print(f"Repeat {repeat_i+1}/{n_repeats}: CNN={single_split_accuracies_50['CNN'][-1]:.1f}%, "
          f"LR={single_split_accuracies_50['LR'][-1]:.1f}%")

    # Save after every repeat, given our history of session drops
    with open("/kaggle/working/repeated_single_splits_50.pkl", "wb") as fh:
        pickle.dump(single_split_accuracies_50, fh)

print("\nDone. Please download repeated_single_splits_50.pkl immediately.")

Repeat 1/50: CNN=64.7%, LR=63.8%
Repeat 2/50: CNN=86.9%, LR=86.9%
Repeat 3/50: CNN=56.9%, LR=67.6%
Repeat 4/50: CNN=54.4%, LR=69.9%
Repeat 5/50: CNN=52.4%, LR=47.6%
Repeat 6/50: CNN=61.9%, LR=61.9%
Repeat 7/50: CNN=31.9%, LR=42.5%
Repeat 8/50: CNN=51.9%, LR=40.4%
Repeat 9/50: CNN=52.9%, LR=66.7%
Repeat 10/50: CNN=52.1%, LR=64.9%
Repeat 11/50: CNN=46.6%, LR=53.4%
Repeat 12/50: CNN=67.3%, LR=65.5%
Repeat 13/50: CNN=84.9%, LR=85.8%
Repeat 14/50: CNN=65.1%, LR=60.6%
Repeat 15/50: CNN=48.6%, LR=73.0%
Repeat 16/50: CNN=82.4%, LR=73.5%
Repeat 17/50: CNN=58.9%, LR=58.9%
Repeat 18/50: CNN=51.0%, LR=50.0%
Repeat 19/50: CNN=62.3%, LR=70.8%
Repeat 20/50: CNN=78.1%, LR=74.3%
Repeat 21/50: CNN=72.5%, LR=61.5%
Repeat 22/50: CNN=82.7%, LR=81.7%
Repeat 23/50: CNN=68.5%, LR=71.0%
Repeat 24/50: CNN=58.0%, LR=64.0%
Repeat 25/50: CNN=70.4%, LR=77.8%
Repeat 26/50: CNN=69.8%, LR=71.9%
Repeat 27/50: CNN=62.0%, LR=53.3%
Repeat 28/50: CNN=55.6%, LR=59.3%
Repeat 29/50: CNN=77.0%, LR=73.0%
Repeat 30/50: CNN=56.2%